# Aula 05 - Notebook: Formas Normais (FND/FNC) e Otimização de Expressões Lógicas

**Disciplina:** ECAA08 — Automática (2026.2) — UNIFEI  
**Projeto:** SCADA-Core Automática / Linha de Produção de Paçoca  
**Equipe:** Grupo 7  
**Área:** Engenharia de Controle e Automação & Matemática Discreta  

---

## 1. Objetivos do Notebook

1. **Extração Canônica:** Implementar o motor algorítmico capaz de extrair a **Forma Normal Disjuntiva (FND / Soma de Mintermos)** e a **Forma Normal Conjuntiva (FNC / Produto de Maxtermos)** a partir de qualquer função booleana de supervisão.
2. **Aplicação na Fábrica de Paçoca:** Mapear e otimizar regras de segurança do **Forno de Torra (Setor 200)** e da **Esteira de Rejeição e Embalagem (Setor 400)**.
3. **Prova Formal de Equivalência:** Validar exaustivamente que as expressões simplificadas por álgebra booleana mantêm $100\%$ de equivalência lógica com as formulações brutas para todas as $2^n$ combinações de entradas.
4. **Benchmark de Tempo Real:** Mensurar o ganho de desempenho (*speedup* e latência de scan) obtido pela redução de termos booleanos em controladores industriais.

In [ ]:
import itertools
import time
from typing import List, Callable, Dict, Any
import pandas as pd

def extrair_formas_normais(variaveis: List[str], funcao: Callable[[Dict[str, bool]], bool]) -> Dict[str, Any]:
    """
    Varre exaustivamente os 2^n estados do espaco booleano para sintetizar
    a Forma Normal Disjuntiva (FND) e a Forma Normal Conjuntiva (FNC).
    """
    mintermos = []
    maxtermos = []
    tabela_verdade = []
    
    for combo in itertools.product([False, True], repeat=len(variaveis)):
        estado = dict(zip(variaveis, combo))
        resultado = funcao(estado)
        
        tabela_verdade.append({**estado, "Saida": resultado})
        
        if resultado:
            # Mintermo para FND (onde a saida e 1)
            termo = [f"{v}" if estado[v] else f"¬{v}" for v in variaveis]
            mintermos.append("(" + " ∧ ".join(termo) + ")")
        else:
            # Maxtermo para FNC (onde a saida e 0)
            termo = [f"¬{v}" if estado[v] else f"{v}" for v in variaveis]
            maxtermos.append("(" + " ∨ ".join(termo) + ")")
            
    fnd_str = " ∨ ".join(mintermos) if mintermos else "FALSO (0)"
    fnc_str = " ∧ ".join(maxtermos) if maxtermos else "VERDADEIRO (1)"
    
    return {
        "Total_Mintermos": len(mintermos),
        "Total_Maxtermos": len(maxtermos),
        "FND": fnd_str,
        "FNC": fnc_str,
        "Tabela": pd.DataFrame(tabela_verdade)
    }

print("Motor algoritmico de Formas Normais (FND/FNC) inicializado com sucesso.")

## 2. Setor 200: Válvula de Gás do Forno de Torra (`XV-201`)

Na torra de amendoim, a válvula de combustível `XV-201` ($v_1$) depende das variáveis:
* `f2`: Chave de Fluxo da Exaustão OK (`FS-201`)
* `c1`: Chama do Queimador Detectada (`TS-201`)
* `p_gas_low`: Pressão Baixa de Combustível (`PS-201`)
* `bypass`: Chave de Purga / Partida Assistida

### Comparação Matemática:
* **Não Otimizada (Bruta):**
  $$v_1 = (f_2 \land c_1 \land \neg p_{gas\_low}) \lor (f_2 \land c_1 \land p_{gas\_low} \land \text{Bypass}) \lor (f_2 \land \neg f_2 \land c_1)$$
* **Otimizada por Álgebra Booleana:**
  $$v_{1\_\text{otimizado}} = f_2 \land c_1 \land (\neg p_{gas\_low} \lor \text{Bypass})$$

In [ ]:
vars_gas = ['f2', 'c1', 'p_gas_low', 'bypass']

# Expressao Nao Otimizada (Contem contradicao f2 and not f2 e termos redundantes)
def gas_forno_nao_otimizado(st: Dict[str, bool]) -> bool:
    f2 = st['f2']
    c1 = st['c1']
    p_gas_low = st['p_gas_low']
    bypass = st['bypass']
    
    t1 = f2 and c1 and (not p_gas_low)
    t2 = f2 and c1 and p_gas_low and bypass
    t3 = f2 and (not f2) and c1  # Termo contraditorio: f2 and not f2 == False
    return t1 or t2 or t3

# Expressao Simplificada via Leis da Algebra Booleana
def gas_forno_otimizado(st: Dict[str, bool]) -> bool:
    return st['f2'] and st['c1'] and ((not st['p_gas_low']) or st['bypass'])

# Sintese das Formas Normais Canonicas
analise_forno = extrair_formas_normais(vars_gas, gas_forno_nao_otimizado)

print(f"=== SINTESE CANONICA: FORNO DE TORRA (SETOR 200) ===")
print(f"Total de Mintermos (Estados Ativos - FND): {analise_forno['Total_Mintermos']}")
print(f"Total de Maxtermos (Clausulas de Bloqueio - FNC): {analise_forno['Total_Maxtermos']}")
print("\nForma Normal Disjuntiva Canônica (FND / Soma de Produtos):")
print(analise_forno['FND'])
print("\nForma Normal Conjuntiva Canônica (FNC / Produto de Cláusulas):")
print(analise_forno['FNC'])

display(analise_forno['Tabela']) if 'display' in globals() else print(analise_forno['Tabela'])

## 3. Setor 400: Sistema de Rejeição Pneumática (`XV-401`)

Na linha de compactação e embalagem de paçocas, a válvula solenóide de ejeção `XV-401` ($v_3$) é acionada pelas variáveis:
* `i1`: Câmera Óptica / IA detecta produto quebrado (`VS-401`)
* `d1`: Detector de Metais acusa contaminação (`MD-401`)
* `p_air`: Pressostato de Ar Comprimido OK ($P > 6\text{ bar}$, `PS-402`)

### Comparação Matemática:
* **Não Otimizada:**
  $$v_3 = (i_1 \land p_{air}) \lor (d_1 \land p_{air}) \lor (i_1 \land d_1 \land p_{air})$$
* **Otimizada por Absorção:**
  $$v_{3\_\text{otimizado}} = (i_1 \lor d_1) \land p_{air}$$

In [ ]:
vars_rejeito = ['i1', 'd1', 'p_air']

# Expressao Nao Otimizada
def rejeicao_nao_otimizada(st: Dict[str, bool]) -> bool:
    i1, d1, p_air = st['i1'], st['d1'], st['p_air']
    t1 = i1 and p_air
    t2 = d1 and p_air
    t3 = i1 and d1 and p_air  # Termo absorvido
    return t1 or t2 or t3

# Expressao Otimizada
def rejeicao_otimizada(st: Dict[str, bool]) -> bool:
    return (st['i1'] or st['d1']) and st['p_air']

analise_rejeito = extrair_formas_normais(vars_rejeito, rejeicao_nao_otimizada)
print(f"=== SINTESE CANONICA: SISTEMA DE REJEICAO (SETOR 400) ===")
print(f"Total de Mintermos (FND): {analise_rejeito['Total_Mintermos']}")
print(f"Total de Maxtermos (FNC): {analise_rejeito['Total_Maxtermos']}")
print("\nFND Canônica:")
print(analise_rejeito['FND'])
print("\nFNC Canônica:")
print(analise_rejeito['FNC'])

## 4. Prova Computacional de Equivalência Lógica

Para garantir que a simplificação analítica não alterou a resposta de segurança da planta, provamos formalmente que:
$$\forall \mathbf{x} \in \{0, 1\}^n, f_{\text{bruta}}(\mathbf{x}) \iff f_{\text{otimizada}}(\mathbf{x})$$

In [ ]:
# 1. Validacao do Forno de Torra (Setor 200)
equiv_forno = True
for combo in itertools.product([False, True], repeat=len(vars_gas)):
    st = dict(zip(vars_gas, combo))
    if gas_forno_nao_otimizado(st) != gas_forno_otimizado(st):
        equiv_forno = False
        break

assert equiv_forno, "Erro: A expressao do forno nao e logicamente equivalente!"
print("[OK] Prova Formal Forno: 100% de equivalencia em todos os 16 estados.")

# 2. Validacao do Sistema de Rejeicao (Setor 400)
equiv_rejeito = True
for combo in itertools.product([False, True], repeat=len(vars_rejeito)):
    st = dict(zip(vars_rejeito, combo))
    if rejeicao_nao_otimizada(st) != rejeicao_otimizada(st):
        equiv_rejeito = False
        break

assert equiv_rejeito, "Erro: A expressao de rejeicao nao e logicamente equivalente!"
print("[OK] Prova Formal Rejeicao: 100% de equivalencia em todos os 8 estados.")

## 5. Benchmark de Desempenho e Latência de Scan

Avaliamos o tempo de execução executando $500.000$ ciclos de varredura (*scan iterations*) comparando a formulação não otimizada com a otimizada.

In [ ]:
N_CICLOS = 500000
estado_teste = {'f2': True, 'c1': True, 'p_gas_low': False, 'bypass': False}

# Medicao Nao Otimizado
t0 = time.time()
for _ in range(N_CICLOS):
    _ = gas_forno_nao_otimizado(estado_teste)
t_nao_otimizado = time.time() - t0

# Medicao Otimizado
t0 = time.time()
for _ in range(N_CICLOS):
    _ = gas_forno_otimizado(estado_teste)
t_otimizado = time.time() - t0

speedup = t_nao_otimizado / t_otimizado

df_bench = pd.DataFrame([
    {
        "Implementação": "Não Otimizada (SOP Bruto / Redundante)",
        "Operações Lógicas": "9",
        "Tempo Total (s)": f"{t_nao_otimizado:.4f}",
        "Tempo por Scan (ns)": f"{(t_nao_otimizado / N_CICLOS) * 1e9:.2f}",
        "Speedup": "1.00x"
    },
    {
        "Implementação": "Otimizada (Álgebra Booleana)",
        "Operações Lógicas": "3",
        "Tempo Total (s)": f"{t_otimizado:.4f}",
        "Tempo por Scan (ns)": f"{(t_otimizado / N_CICLOS) * 1e9:.2f}",
        "Speedup": f"{speedup:.2f}x"
    }
])

print("=== BENCHMARK DE PERFORMANCE NO SCADA-CORE ===")
display(df_bench) if 'display' in globals() else print(df_bench.to_string(index=False))

## 6. Conclusões de Engenharia

1. **Representação Canônica:** O algoritmo implementado extrai a FND (ideal para mapear permissivos paralelos) e a FNC (ideal para matrizes de restrição e segurança industrial).
2. **Eliminação de Contradições:** O descarte de termos contraditórios ($f_2 \land \neg f_2 \equiv 0$) e a aplicação da absorção mista ($
eg A \lor (A \land B) \equiv \neg A \lor B$) reduziu o custo operacional de $9$ para $3$ operações lógicas.
3. **Garantia de Equivalência e Desempenho:** Provamos que a integridade da lógica de segurança foi preservada em $100\%$ dos estados operacionais, com aceleração expressiva no tempo de resposta do scan cycle.